# AI Trade Review v1

这个 Notebook 用于把交易日志和 Trader AI Dashboard 的市场环境结合起来，生成交易复盘。

- 没有 OpenAI API Key：先生成离线规则复盘。
- 输入 OpenAI API Key：生成 AI 交易教练复盘。
- API Key 只在本次 Colab 会话中临时保存，不会写入仓库。

In [ ]:
# Colab already pins pandas/numpy for its runtime. Do not upgrade them here.
%pip -q install -U openai yfinance

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from getpass import getpass

REPO_URL = 'https://github.com/cottonlyz-coder/trader-ai-dashboard.git'
COLAB_ROOT = Path('/content/trader-ai-dashboard')
PROJECT_ROOT = COLAB_ROOT if (COLAB_ROOT / 'src').exists() else None

if PROJECT_ROOT is None:
    print('正在下载 Trader AI Dashboard 项目代码...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(COLAB_ROOT)], check=True)
    PROJECT_ROOT = COLAB_ROOT
else:
    print('正在同步最新项目代码...')
    subprocess.run(['git', '-C', str(COLAB_ROOT), 'pull', '--ff-only'], check=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

api_key = getpass('OpenAI API Key（可留空，留空则使用离线复盘）: ')
if api_key.strip():
    os.environ['OPENAI_API_KEY'] = api_key.strip()
os.environ.setdefault('OPENAI_MODEL', 'gpt-5-mini')
print(f'项目代码已就绪：{PROJECT_ROOT}')

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from src.ai_review import (
    generate_offline_trade_review,
    review_trades_with_openai,
    summarize_trade_journal,
)
from src.data_loader import load_market_data
from src.indicators import calculate_market_metrics
from src.regime import classify_market_regime

# 先使用样例日志。之后可以把这里改成你自己的 CSV 路径。
TRADE_LOG_PATH = PROJECT_ROOT / 'data' / 'trade_journal_sample.csv'
trades = pd.read_csv(TRADE_LOG_PATH)
display(Markdown('## 交易日志'))
display(trades)
display(Markdown('## 交易统计'))
display(summarize_trade_journal(trades).style.format({'Total PnL': '{:,.2f}', 'Win Rate (%)': '{:.1f}%', 'Average PnL': '{:,.2f}', 'Average R Multiple': '{:.2f}', 'Largest Loss': '{:,.2f}', 'Largest Win': '{:,.2f}'}))

In [ ]:
prices, symbol_info, messages = load_market_data(period='2y')
metrics = calculate_market_metrics(prices)
regime = classify_market_regime(metrics)
display(Markdown(f'## 当前市场状态：**{regime.label}** | 分数 {regime.score:+d}'))
for reason in regime.reasons:
    print('-', reason)

if 'OPENAI_API_KEY' in os.environ:
    review = review_trades_with_openai(trades, metrics, regime)
else:
    review = generate_offline_trade_review(trades, metrics, regime)

display(Markdown(review))